# RAIL Score SDK — Complete Guide

This notebook demonstrates every feature of the RAIL Score SDK v2.2.1, including:
- Core evaluation (basic & deep modes)
- Custom dimensions, weights, domains & contexts
- Safe regeneration (auto & external modes)
- Regulatory compliance checking (single & multi-framework)
- Async client with caching & concurrency
- Multi-turn sessions with policy enforcement
- LLM provider integrations (OpenAI, Anthropic, Gemini)
- OpenTelemetry observability & structured compliance logging
- Error handling patterns

All code cells are verified against the live RAIL Score API.

## 1. Installation & Setup

In [ ]:
# Install the SDK with all optional dependencies
# pip install rail-score-sdk[telemetry,integrations]

# For this notebook, we load environment variables from .env
import os
from dotenv import load_dotenv

load_dotenv()

RAIL_API_KEY = os.environ["RAIL_SCORE_API"]
print(f"API key loaded: {RAIL_API_KEY[:10]}...")

## 2. Telemetry Configuration (OpenTelemetry)

Configure vendor-neutral observability with org/project/environment grouping. Uses console exporter for this guide — switch to `exporter="otlp"` for production.

In [ ]:
from rail_score_sdk.telemetry import RAILTelemetry

telemetry = RAILTelemetry(
    org_id="my-org",
    project_id="chatbot-v2",
    environment="notebook-demo",
    exporter="console",       # "otlp" for production, "console" for debugging
    enable_traces=True,
    enable_metrics=True,
    enable_logs=True,
)
print(f"Telemetry configured: org={telemetry.org_id}, project={telemetry.project_id}")

# For production with an OTLP collector:
# telemetry = RAILTelemetry(
#     org_id="my-org",
#     project_id="chatbot-v2",
#     environment="production",
#     exporter="otlp",
#     endpoint="localhost:4317",
#     protocol="grpc",
#     headers={"Authorization": "Bearer <collector-token>"},
# )

## 3. Health Check

In [ ]:
from rail_score_sdk import RailScoreClient

# Create client with telemetry enabled (traces every API call automatically)
client = RailScoreClient(api_key=RAIL_API_KEY, telemetry=telemetry)

# Health check — no authentication required
health = client.health()
print(f"Status:  {health.status}")
print(f"Service: {health.service}")

## 4. Basic Evaluation

Score content across 8 RAIL dimensions: fairness, safety, reliability, transparency, privacy, accountability, inclusivity, user_impact.

In [ ]:
result = client.eval(
    content="Regular exercise like walking 30 minutes daily improves cardiovascular "
            "health and reduces risk of chronic diseases. Always consult your doctor "
            "before starting a new exercise program.",
    mode="basic",
)

print(f"Overall Score:      {result.rail_score.score}/10")
print(f"Confidence:         {result.rail_score.confidence}")
print(f"Summary:            {result.rail_score.summary}\n")

print("Dimension Scores:")
for dim, ds in result.dimension_scores.items():
    print(f"  {dim:20s}  score={ds.score:.1f}  confidence={ds.confidence:.2f}")

## 5. Deep Evaluation with Explanations

Deep mode provides per-dimension explanations for every score.

In [ ]:
deep_result = client.eval(
    content="Regular exercise like walking 30 minutes daily improves cardiovascular "
            "health and reduces risk of chronic diseases. Always consult your doctor "
            "before starting a new exercise program.",
    mode="deep",
)

print(f"Overall Score: {deep_result.rail_score.score}/10\n")

for dim, ds in deep_result.dimension_scores.items():
    print(f"--- {dim} (score={ds.score:.1f}) ---")
    if ds.explanation:
        print(f"  {ds.explanation[:200]}")
    if ds.issues:
        print(f"  Issues: {ds.issues}")
    print()

## 6. Custom Dimensions & Weights

Evaluate only specific dimensions, and apply custom weights (must sum to 100).

In [ ]:
content = (
    "Our AI system processes user data to provide personalized recommendations. "
    "We use encryption for data storage and clearly explain our data usage "
    "policies to users. All decisions can be traced and audited."
)

# Evaluate only safety and reliability
selected = client.eval(content=content, dimensions=["safety", "reliability"])
print("Selected dimensions:", list(selected.dimension_scores.keys()))
for dim, ds in selected.dimension_scores.items():
    print(f"  {dim}: {ds.score:.1f}")

print()

# Custom weights (must sum to 100)
weighted = client.eval(
    content=content,
    weights={
        "safety": 30, "reliability": 20, "fairness": 15,
        "transparency": 10, "privacy": 10, "accountability": 5,
        "inclusivity": 5, "user_impact": 5,
    },
)
print(f"Weighted overall score: {weighted.rail_score.score}/10")

## 7. Domain-Specific Evaluation

Tailor scoring to specific domains (healthcare, finance, legal, education, code) and use cases.

In [ ]:
# Healthcare domain with chatbot use case and additional context
healthcare_result = client.eval(
    content=content,
    mode="deep",
    domain="healthcare",
    usecase="chatbot",
    context="medical chatbot for elderly patients",
    include_explanations=True,
    include_issues=True,
    include_suggestions=True,
)

print(f"Healthcare Score: {healthcare_result.rail_score.score}/10")
print(f"Summary: {healthcare_result.rail_score.summary}")

if healthcare_result.issues:
    print(f"\nIssues found: {len(healthcare_result.issues)}")
    for issue in healthcare_result.issues[:3]:
        print(f"  [{issue.dimension}] {issue.description}")

if healthcare_result.improvement_suggestions:
    print(f"\nSuggestions:")
    for s in healthcare_result.improvement_suggestions[:3]:
        print(f"  - {s}")

## 8. Safe Regeneration — Auto Mode (RAIL_Safe_LLM)

The server iteratively regenerates content until RAIL thresholds are met.

In [ ]:
risky_content = (
    "Our AI collects all user data including browsing history and shares it "
    "with partners. We don't explain how decisions are made. Trust us, we know best."
)

regen_result = client.safe_regenerate(
    content=risky_content,
    regeneration_model="RAIL_Safe_LLM",
    max_regenerations=2,
)

print(f"Status:           {regen_result.status}")
print(f"Credits consumed: {regen_result.credits_consumed}")

if regen_result.best_content:
    print(f"\nImproved content:\n  {regen_result.best_content[:300]}")

if regen_result.iteration_history:
    print(f"\nIteration history ({len(regen_result.iteration_history)} iterations):")
    for it in regen_result.iteration_history:
        print(f"  Iteration {it.iteration}: {it.content[:80]}...")

if regen_result.credits_breakdown:
    cb = regen_result.credits_breakdown
    print(f"\nCredits: eval={cb.evaluations}, regen={cb.regenerations}, total={cb.total}")

## 9. Safe Regeneration — External Mode

In external mode, the API returns a prompt — you regenerate with your own LLM, then continue the session.

In [ ]:
# Step 1: Start external session
ext_result = client.safe_regenerate(
    content=risky_content,
    regeneration_model="external",
)

print(f"Status:     {ext_result.status}")
print(f"Session ID: {ext_result.session_id}")
print(f"Rail Prompt (system): {ext_result.rail_prompt.system_prompt[:100]}...")
print(f"Rail Prompt (user):   {ext_result.rail_prompt.user_prompt[:100]}...")

# Step 2: Regenerate content yourself (using the rail_prompt with your LLM)
improved_content = (
    "Our AI system processes user data responsibly with full encryption and "
    "clear consent mechanisms. All data usage is transparent and users maintain "
    "control over their information. Decisions are explainable and auditable."
)

# Step 3: Continue the session with your regenerated content
cont_result = client.safe_regenerate_continue(
    session_id=ext_result.session_id,
    regenerated_content=improved_content,
)

print(f"\nContinue status: {cont_result.status}")
if cont_result.status in ("passed", "max_iterations_reached"):
    print(f"Best content: {cont_result.best_content[:200]}...")

## 10. Compliance Check — Single Framework

Evaluate content against regulatory frameworks: GDPR, CCPA, HIPAA, EU AI Act, India DPDP, India AI Governance.

In [ ]:
compliance_content = (
    "Our AI system processes personal health records to make automated decisions "
    "about patient care. We collect extensive user data including medical history, "
    "genetic information, and behavioral patterns."
)

gdpr_result = client.compliance_check(
    content=compliance_content,
    framework="gdpr",
    context={"domain": "healthcare", "data_types": ["health_records", "genetic_data"]},
)

print(f"Framework:  {gdpr_result.framework} (v{gdpr_result.framework_version})")
print(f"Score:      {gdpr_result.compliance_score.score}/10")
print(f"Label:      {gdpr_result.compliance_score.label}")
print(f"Confidence: {gdpr_result.compliance_score.confidence}")
print(f"Summary:    {gdpr_result.compliance_score.summary[:200]}")

print(f"\nRequirements: {gdpr_result.requirements_checked} checked, "
      f"{gdpr_result.requirements_passed} passed, "
      f"{gdpr_result.requirements_failed} failed")

if gdpr_result.issues:
    print(f"\nTop issues ({len(gdpr_result.issues)} total):")
    for issue in gdpr_result.issues[:3]:
        print(f"  [{issue.severity}] {issue.description[:100]}")
        print(f"    Article: {issue.article} | Remediation: {issue.remediation_effort}")

## 11. Compliance Check — Multi-Framework

Evaluate against multiple frameworks in a single call (up to 5). Returns a cross-framework summary.

In [ ]:
multi_result = client.compliance_check(
    content=compliance_content,
    frameworks=["gdpr", "ccpa"],
)

# Cross-framework summary
summary = multi_result.cross_framework_summary
print(f"Frameworks evaluated: {summary.frameworks_evaluated}")
print(f"Average score:        {summary.average_score}/10")
print(f"Weakest framework:    {summary.weakest_framework} ({summary.weakest_score}/10)")

# Per-framework results
print("\nPer-framework breakdown:")
for fw_name, fw_result in multi_result.results.items():
    print(f"  {fw_name}: score={fw_result.compliance_score.score}, "
          f"label={fw_result.compliance_score.label}, "
          f"passed={fw_result.requirements_passed}/{fw_result.requirements_checked}")

## 12. Async Client Usage

The async client returns raw dicts, supports built-in caching (5-min TTL), and automatic retries.

In [ ]:
import asyncio
from rail_score_sdk import AsyncRAILClient

async def async_demo():
    async with AsyncRAILClient(api_key=RAIL_API_KEY) as aclient:
        # Health check
        health = await aclient.health()
        print(f"Health: {health['status']}")

        # Basic eval (returns dict)
        result = await aclient.eval(
            "Regular exercise improves cardiovascular health and reduces risk "
            "of chronic diseases. Consult your doctor before starting new routines.",
            mode="basic",
        )
        print(f"Score: {result['rail_score']['score']}/10")

        # Caching — second call is instant
        result2 = await aclient.eval(
            "Regular exercise improves cardiovascular health and reduces risk "
            "of chronic diseases. Consult your doctor before starting new routines.",
            mode="basic",
        )
        print(f"Cached: {result2.get('from_cache', 'N/A')}")

        # Concurrent evaluations with asyncio.gather
        texts = [
            "AI systems should be fair and unbiased in their decision making.",
            "Data privacy is a fundamental right that must be protected.",
            "Transparent algorithms build trust with users and stakeholders.",
        ]
        results = await asyncio.gather(
            *[aclient.eval(t, mode="basic") for t in texts]
        )
        scores = [r["rail_score"]["score"] for r in results]
        print(f"Concurrent scores: {scores}")

await async_demo()

## 13. Multi-Turn Session

`RAILSession` tracks conversation history, running averages, and applies policies per-turn.

In [ ]:
from rail_score_sdk import RAILSession

async def session_demo():
    async with RAILSession(
        api_key=RAIL_API_KEY,
        threshold=7.0,
        policy="log_only",
        mode="basic",
        deep_every_n=2,  # Run deep mode every 2nd turn
    ) as session:
        turns = [
            ("What is a healthy diet?",
             "A healthy diet includes fruits, vegetables, whole grains, and lean "
             "proteins. It's important to maintain balanced nutrition and stay "
             "hydrated. Consult a nutritionist for personalized advice."),
            ("What about exercise?",
             "Regular exercise of 30 minutes daily helps maintain cardiovascular "
             "health. Activities like walking, swimming, and cycling are great "
             "options. Always warm up before exercising."),
            ("How about sleep?",
             "Adults need 7-9 hours of quality sleep per night. Maintain a "
             "consistent sleep schedule and create a comfortable sleep "
             "environment. Avoid screens before bedtime."),
        ]

        for user_msg, response in turns:
            result = await session.evaluate_turn(
                user_message=user_msg,
                assistant_response=response,
            )
            print(f"Turn: {user_msg[:30]}... → score={result.score:.1f}")

        # Pre-screen user input (does NOT add to history)
        input_eval = await session.evaluate_input("What medications should I take?")
        print(f"\nInput pre-screen score: {input_eval.score:.1f}")

        # Session summary
        print(f"\nSession Summary:")
        print(f"  History length: {len(session.history)}")
        print(f"  Average score:  {session.average_score:.2f}")
        print(f"  Lowest score:   {session.lowest_score:.2f}")
        print(f"  Full summary:   {session.scores_summary()}")

await session_demo()

## 14. Policy Engine (LOG_ONLY, BLOCK, REGENERATE)

Policies control what happens when a response scores below threshold.

In [ ]:
from rail_score_sdk import Policy, RAILBlockedError

async def policy_demo():
    # LOG_ONLY: Always passes through, just attaches scores
    async with RAILSession(
        api_key=RAIL_API_KEY, threshold=9.0, policy=Policy.LOG_ONLY
    ) as session:
        result = await session.evaluate_turn(
            user_message="Tell me about data",
            assistant_response="Data is stored in databases and can be queried. "
                "We process information using standard methods.",
        )
        print(f"LOG_ONLY: score={result.score:.1f}, threshold_met={result.threshold_met}")

    # BLOCK: Raises RAILBlockedError when score < threshold
    async with RAILSession(
        api_key=RAIL_API_KEY, threshold=9.5, policy=Policy.BLOCK
    ) as session:
        try:
            result = await session.evaluate_turn(
                user_message="Tell me something",
                assistant_response="Here is some basic information. It might be "
                    "helpful, perhaps, maybe, I think.",
            )
            print(f"BLOCK: passed with score={result.score:.1f}")
        except RAILBlockedError as e:
            print(f"BLOCK: Blocked! score={e.score:.1f}, threshold={e.threshold}")

    # REGENERATE: Auto-regenerates via safe_regenerate when below threshold
    async with RAILSession(
        api_key=RAIL_API_KEY, threshold=7.0, policy=Policy.REGENERATE
    ) as session:
        result = await session.evaluate_turn(
            user_message="How do you handle data?",
            assistant_response="Data handling is done somehow. We do stuff with "
                "user info. Trust the process and don't ask too many questions.",
        )
        print(f"REGENERATE: score={result.score:.1f}, "
              f"was_regenerated={result.was_regenerated}")

await policy_demo()

## 15. OpenAI Integration (RAILOpenAI)

Wraps OpenAI's `AsyncOpenAI` client — every chat completion is automatically RAIL-evaluated.

In [ ]:
from rail_score_sdk.integrations import RAILOpenAI

async def openai_demo():
    rail_openai = RAILOpenAI(
        openai_api_key=os.environ["OPENAI_API_KEY"],
        rail_api_key=RAIL_API_KEY,
        rail_threshold=7.0,
        rail_policy="log_only",
        rail_mode="basic",
    )

    response = await rail_openai.chat_completion(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Explain the importance of data privacy in healthcare AI systems"}],
    )

    print(f"Content:     {response.content[:200]}...")
    print(f"RAIL Score:  {response.rail_score}/10")
    print(f"Confidence:  {response.rail_confidence}")
    print(f"Threshold:   {'Met' if response.threshold_met else 'Not met'}")
    print(f"Model:       {response.model}")
    print(f"Tokens:      {response.usage}")

await openai_demo()

## 16. Anthropic Integration (RAILAnthropic)

Wraps Anthropic's `AsyncAnthropic` client with automatic RAIL evaluation.

In [ ]:
from rail_score_sdk.integrations import RAILAnthropic

async def anthropic_demo():
    rail_anthropic = RAILAnthropic(
        anthropic_api_key=os.environ["ANTHROPIC_API_KEY"],
        rail_api_key=RAIL_API_KEY,
        rail_threshold=7.0,
        rail_policy="log_only",
    )

    response = await rail_anthropic.message(
        model="claude-sonnet-4-5-20250929",
        max_tokens=512,
        messages=[{"role": "user", "content": "What are the ethical considerations for AI in financial services?"}],
    )

    print(f"Content:     {response.content[:200]}...")
    print(f"RAIL Score:  {response.rail_score}/10")
    print(f"Confidence:  {response.rail_confidence}")
    print(f"Model:       {response.model}")

await anthropic_demo()

## 17. Gemini Integration (RAILGemini)

Wraps Google's `google-genai` SDK with automatic RAIL evaluation.

In [ ]:
from rail_score_sdk.integrations import RAILGemini

async def gemini_demo():
    rail_gemini = RAILGemini(
        gemini_api_key=os.environ["GEMINI_API_KEY"],
        rail_api_key=RAIL_API_KEY,
        rail_threshold=7.0,
        rail_policy="log_only",
    )

    response = await rail_gemini.generate(
        model="gemini-2.5-flash",
        contents="Describe best practices for responsible AI deployment in education",
    )

    print(f"Content:     {response.content[:200]}...")
    print(f"RAIL Score:  {response.rail_score}/10")
    print(f"Confidence:  {response.rail_confidence}")
    print(f"Model:       {response.model}")

await gemini_demo()

## 18. Error Handling Patterns

All SDK exceptions inherit from `RailScoreError` and carry `status_code` and `response` attributes.

In [ ]:
from rail_score_sdk.exceptions import (
    RailScoreError,
    AuthenticationError,
    ValidationError,
    SessionExpiredError,
)

# 1. Invalid API key → AuthenticationError (401)
try:
    bad_client = RailScoreClient(api_key="invalid_key_12345")
    bad_client.eval("Test content for authentication validation purposes.", mode="basic")
except AuthenticationError as e:
    print(f"AuthenticationError: status={e.status_code}, msg={e.message[:60]}")

# 2. Content too short → ValidationError (400)
try:
    client.eval("Hi", mode="basic")
except ValidationError as e:
    print(f"ValidationError (short): status={e.status_code}, msg={e.message[:60]}")

# 3. Invalid mode → ValidationError (400)
try:
    client.eval("Sufficiently long test content for validation.", mode="invalid_mode")
except ValidationError as e:
    print(f"ValidationError (mode): status={e.status_code}, msg={e.message[:60]}")

# 4. Weights not summing to 100 → ValidationError (400)
try:
    client.eval("Sufficiently long test content for validation.",
                weights={"safety": 50, "reliability": 20})
except ValidationError as e:
    print(f"ValidationError (weights): status={e.status_code}, msg={e.message[:60]}")

# 5. No framework → ValueError (client-side)
try:
    client.compliance_check("Content for compliance testing purposes.")
except ValueError as e:
    print(f"ValueError (no framework): {e}")

# 6. Both framework and frameworks → ValueError (client-side)
try:
    client.compliance_check("Content here.", framework="gdpr", frameworks=["ccpa"])
except ValueError as e:
    print(f"ValueError (both): {e}")

# 7. Fake session → SessionExpiredError (410)
try:
    client.safe_regenerate_continue(
        session_id="sr_fake_session_12345",
        regenerated_content="Improved content for the expired session test.",
    )
except SessionExpiredError as e:
    print(f"SessionExpiredError: status={e.status_code}, msg={e.message[:60]}")

# 8. Exception hierarchy
print(f"\nAll inherit from RailScoreError: "
      f"Auth={issubclass(AuthenticationError, RailScoreError)}, "
      f"Val={issubclass(ValidationError, RailScoreError)}, "
      f"Sess={issubclass(SessionExpiredError, RailScoreError)}")

## 19. Observability & Structured Compliance Logging

Use `ComplianceLogger` to emit structured OTEL log records for compliance results — grouped by org, project, and environment.

In [ ]:
from rail_score_sdk.telemetry import ComplianceLogger

comp_logger = ComplianceLogger(telemetry)

# Log single-framework compliance result (uses the gdpr_result from Section 10)
print("--- Single-framework compliance log ---")
comp_logger.log_compliance_result(gdpr_result, content_preview=compliance_content[:50])

# Log multi-framework compliance result (uses the multi_result from Section 11)
print("\n--- Multi-framework compliance log ---")
comp_logger.log_multi_compliance_result(multi_result)

print("\nStructured logs emitted to console exporter.")
print("In production (exporter='otlp'), these flow to your observability backend,")
print("grouped by rail.org_id, rail.project_id, rail.environment.")

## 20. Cleanup & Shutdown

In [ ]:
# Flush all pending telemetry data and release resources
telemetry.shutdown()
print("Telemetry shutdown complete — all pending spans, metrics, and logs flushed.")